<a href="https://colab.research.google.com/github/unknown-arc/Machine-Learning-from-Scratch/blob/main/Decision_Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Decision Tree Classification**


# **1. Load Datasets**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/ML from scratch/Data-Assests/titanic.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **2. Import Libraries**

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

# **3. Dataset Pre-Process**

## **A. Preview Dataset**

In [1]:
df.head()

NameError: name 'df' is not defined

In [2]:
df.tail()

NameError: name 'df' is not defined

In [3]:
df.info()

NameError: name 'df' is not defined

## **B. Removing unnecessary columns**

In [ ]:
df = df.drop(['PassengerId', 'Name', 'SibSp', 'Parch', 'Ticket', 'Cabin', 'Embarked'], axis=1)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Survived  891 non-null    int64  
 1   Pclass    891 non-null    int64  
 2   Sex       891 non-null    object 
 3   Age       714 non-null    float64
 4   Fare      891 non-null    float64
dtypes: float64(2), int64(2), object(1)
memory usage: 34.9+ KB


## **C. Preprocess the dataset**

In [ ]:
print(df.isnull().sum())
df['Age'].fillna(df['Age'].median(), inplace=True)

Survived      0
Pclass        0
Sex           0
Age         177
Fare          0
dtype: int64


/tmp/ipython-input-2035411855.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)


In [ ]:
df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})


## **D. Define the target variable as Survived.**


In [ ]:
X = df.drop('Survived', axis=1)
y = df['Survived']


# **4. Spliting dataset into training and testing sets**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)


In [ ]:
print(X_train.shape)
print(X_test.shape)


(712, 4)
(179, 4)


# **5. DECISION TREE MODAL from Scratch.**

## **A. MODAL**

In [ ]:
class Node:
    def __init__(self, feature_idx=None, threshold=None, info_gain=None,
                 left=None, right=None, value=None):

        self.feature_idx = feature_idx
        self.threshold = threshold
        self.info_gain = info_gain
        self.left = left
        self.right = right
        self.value = value  # For leaf nodes


In [ ]:
class DecisionTree:
    def __init__(self, min_samples_split=2, max_depth=2):

        self.min_samples_split = min_samples_split
        self.max_depth = max_depth

    def build_tree(self, dataset, curr_depth=0):
        X, y = dataset[:, :-1], dataset[:, -1]
        n_samples, n_features = X.shape

        if n_samples >= self.min_samples_split and curr_depth <= self.max_depth:
            best_split = self.best_split(dataset, n_features)

            if best_split["info_gain"] > 0:
                left_node = self.build_tree(best_split["left_dataset"], curr_depth + 1)
                right_node = self.build_tree(best_split["right_dataset"], curr_depth + 1)

                return Node(best_split["feature_idx"], best_split["threshold"], best_split["info_gain"], left_node, right_node)

        leaf_value = Counter(y).most_common(1)[0][0]
        return Node(value=leaf_value)

    def best_split(self, dataset, n_features):
        best_split = {'feature_idx': None, 'threshold': None, 'info_gain': -1, 'left_dataset': None, 'right_dataset': None}

        for feature_idx in range(n_features):
            feature_values = dataset[:, feature_idx]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:
                left_dataset, right_dataset = self.split(dataset, feature_idx, threshold)

                if len(left_dataset) and len(right_dataset):
                    parent_y, left_y, right_y = dataset[:, -1], left_dataset[:, -1], right_dataset[:, -1]

                    info_gain = self.information_gain(parent_y, left_y, right_y)

                    if info_gain > best_split['info_gain']:
                        best_split['feature_idx'] = feature_idx
                        best_split['threshold'] = threshold
                        best_split['info_gain'] = info_gain
                        best_split['left_dataset'] = left_dataset
                        best_split['right_dataset'] = right_dataset

        return best_split

    def split(self, dataset, feature_idx, threshold):
        left_dataset = np.array([row for row in dataset if row[feature_idx] <= threshold])
        right_dataset = np.array([row for row in dataset if row[feature_idx] > threshold])

        return left_dataset, right_dataset

    def information_gain(self, parent_y, left_y, right_y):
        left_weight = len(left_y) / len(parent_y)
        right_weight = len(right_y) / len(parent_y)

        information_gain = self.entropy(parent_y) - (left_weight * self.entropy(left_y) + right_weight * self.entropy(right_y))
        return information_gain

    def entropy(self, y):
        entropy = 0

        class_labels = np.unique(y)
        for class_label in class_labels:
            p = len(y[y == class_label]) / len(y)
            entropy += -p * np.log2(p)

        return entropy

    def fit(self, X, y):
        dataset = np.concatenate([X, y.reshape(-1, 1)], axis=1)
        self.root = self.build_tree(dataset)

    def predict(self, X):
        predictions = [self.predict_class(row, self.root) for row in X]
        return predictions

    def predict_class(self, row, node):
        if node.value != None:
            return node.value

        feature_val = row[node.feature_idx]
        if feature_val <= node.threshold:
            return self.predict_class(row, node.left)
        else:
            return self.predict_class(row, node.right)

    def print_tree(self, node=None, depth=0, indent="|   "):
        prefix = indent * depth

        if node is None:
            node = self.root

        if node.value is not None:
            print(f"{prefix}|--- class: {node.value}")
            return

        feature_label = f"Feature {node.feature_idx}"

        print(f"{prefix}|--- {feature_label} <= {node.threshold}")
        self.print_tree(node.left, depth + 1, indent)

        print(f"{prefix}|--- {feature_label} > {node.threshold}")
        self.print_tree(node.right, depth + 1, indent)

## **B. Make predictions and evaluate the model.**

### Convert to NumPy

In [ ]:
X_train_np = X_train.values
y_train_np = y_train.values
X_test_np = X_test.values
y_test_np = y_test.values


### Train the Model and Make Predictions

In [ ]:
tree = DecisionTree(max_depth=3)
tree.fit(X_train_np, y_train_np)

y_pred = tree.predict(X_test_np)


### Accuracy

In [ ]:
accuracy = np.mean(y_pred == y_test) * 100
print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 79.89%


## **C. Visualize the constructed Decision Tree.**

In [ ]:
tree.print_tree()


|--- Feature 1 <= 0.0
|   |--- Feature 2 <= 6.0
|   |   |--- Feature 0 <= 2.0
|   |   |   |--- class: 1.0
|   |   |--- Feature 0 > 2.0
|   |   |   |--- Feature 3 <= 20.575
|   |   |   |   |--- class: 1.0
|   |   |   |--- Feature 3 > 20.575
|   |   |   |   |--- class: 0.0
|   |--- Feature 2 > 6.0
|   |   |--- Feature 0 <= 1.0
|   |   |   |--- Feature 3 <= 26.0
|   |   |   |   |--- class: 0.0
|   |   |   |--- Feature 3 > 26.0
|   |   |   |   |--- class: 0.0
|   |   |--- Feature 0 > 1.0
|   |   |   |--- Feature 2 <= 32.0
|   |   |   |   |--- class: 0.0
|   |   |   |--- Feature 2 > 32.0
|   |   |   |   |--- class: 0.0
|--- Feature 1 > 0.0
|   |--- Feature 0 <= 2.0
|   |   |--- Feature 2 <= 2.0
|   |   |   |--- Feature 0 <= 1.0
|   |   |   |   |--- class: 0.0
|   |   |   |--- Feature 0 > 1.0
|   |   |   |   |--- class: 1.0
|   |   |--- Feature 2 > 2.0
|   |   |   |--- Feature 2 <= 27.0
|   |   |   |   |--- class: 1.0
|   |   |   |--- Feature 2 > 27.0
|   |   |   |   |--- class: 1.0
|   |---

# **6. Compare & Result**

## **A. Calculate Accuracy, confusion matrix and Classification Report**

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("===== Custom Decision Tree =====")
print("Accuracy:", accuracy_score(y_test_np, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test_np, y_pred))
print("Classification Report:\n", classification_report(y_test_np, y_pred))


===== Custom Decision Tree =====
Accuracy: 0.7988826815642458
Confusion Matrix:
 [[92 13]
 [23 51]]
Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.88      0.84       105
           1       0.80      0.69      0.74        74

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179



## **B. Scikit-learn's DecisionTreeClassifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

sk_tree = DecisionTreeClassifier(
    criterion="entropy",
    max_depth=3,
    random_state=42
)

sk_tree.fit(X_train, y_train)

y_pred_sk = sk_tree.predict(X_test)

print("\n===== Sklearn Decision Tree =====")
print("Accuracy:", np.mean(y_pred_sk == y_test_np) * 100)
print("Confusion Matrix:\n", confusion_matrix(y_test_np, y_pred_sk))
print("Classification Report:\n")
print(classification_report(y_test_np, y_pred_sk))



===== Sklearn Decision Tree =====
Accuracy: 79.88826815642457
Confusion Matrix:
 [[92 13]
 [23 51]]
Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.88      0.84       105
           1       0.80      0.69      0.74        74

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

